In [2]:
from finmind import FinMindClient

client = FinMindClient()

taiex, inst, usd, us_market = client.get_macro_indicators(
    start_date="2025-01-01",
    end_date="2025-03-31",
)

for name, df in {
    "TAIEX": taiex,
    "INST": inst,
    "USD": usd,
    "US": us_market,
}.items():
    print(f"\n===== {name} =====")
    print(df.columns.tolist())
    print(df.dtypes)
    print(df.tail())

2026-09-21 00:05:00.241 | INFO     | FinMind.data.finmind_api:login_by_token:85 - Login success
2026-09-21 00:05:00.287 | INFO     | FinMind.data.finmind_api:login_by_token:85 - Login success
2026-09-21 00:05:00.288 | INFO     | FinMind.data.finmind_api:get_data:166 - download Dataset.TaiwanStockTotalReturnIndex, data_id: TAIEX
2026-09-21 00:05:00.482 | INFO     | FinMind.data.finmind_api:get_data:166 - download Dataset.TaiwanStockTotalInstitutionalInvestors, data_id: 



===== TAIEX =====
['price', 'index_id', 'date']
price       float64
index_id     object
date         object
dtype: object
       price index_id        date
50  49349.28    TAIEX  2025-03-25
51  49321.23    TAIEX  2025-03-26
52  48649.66    TAIEX  2025-03-27
53  47876.79    TAIEX  2025-03-28
54  45866.82    TAIEX  2025-03-31

===== INST =====
['buy', 'date', 'name', 'sell']
buy      int64
date    object
name    object
sell     int64
dtype: object
              buy        date                 name          sell
325   19305443744  2025-03-31       Dealer_Hedging   37620676085
326  149652071600  2025-03-31     Foreign_Investor  187506788804
327    6701089416  2025-03-31          Dealer_self   10422567624
328             0  2025-03-31  Foreign_Dealer_Self             0
329  189785292609  2025-03-31                total  241511383550

===== USD =====
['date', 'currency', 'cash_buy', 'cash_sell', 'spot_buy', 'spot_sell']
date          object
currency      object
cash_buy     float64
cash_sel

In [3]:
from finmind import FinMindClient
from investment_agents.data.repositories.market_regime import (
    MarketRegimeRepository,
)

client = FinMindClient()
repo = MarketRegimeRepository(client)

data = repo.get_data(
    start_date="2025-01-01",
    end_date="2025-03-31",
)

for name, df in data.items():
    print(f"\n===== {name} =====")
    print(df.dtypes)
    print(df.tail())

2026-09-21 00:05:01.985 | INFO     | FinMind.data.finmind_api:get_data:166 - download Dataset.TaiwanStockTotalReturnIndex, data_id: TAIEX
2026-09-21 00:05:02.052 | INFO     | FinMind.data.finmind_api:get_data:166 - download Dataset.TaiwanStockTotalInstitutionalInvestors, data_id: 



===== taiex =====
date     datetime64[ns]
taiex           float64
dtype: object
         date     taiex
50 2025-03-25  49349.28
51 2025-03-26  49321.23
52 2025-03-27  48649.66
53 2025-03-28  47876.79
54 2025-03-31  45866.82

===== institutional =====
date             datetime64[ns]
investor_type            object
buy                       int64
sell                      int64
net_buy                   int64
dtype: object
          date        investor_type           buy          sell      net_buy
270 2025-03-31       Dealer_Hedging   19305443744   37620676085 -18315232341
271 2025-03-31          Dealer_self    6701089416   10422567624  -3721478208
272 2025-03-31  Foreign_Dealer_Self             0             0            0
273 2025-03-31     Foreign_Investor  149652071600  187506788804 -37854717204
274 2025-03-31     Investment_Trust   14126687849    5961351037   8165336812

===== usd_twd =====
date       datetime64[ns]
usd_twd           float64
dtype: object
         date  usd_twd
52

In [4]:
from investment_agents.features.regime import RegimeFeatureService

service = RegimeFeatureService()

panel = service.align_market_data(
    taiex=data["taiex"],
    institutional=data["institutional"],
    usd_twd=data["usd_twd"],
    us_market=data["us_market"],
)

print(
    panel[
        [
            "date",
            "taiex",
            "usd_twd",
            "spy",
            "qqq",
            "soxx",
            "tlt",
            "uup",
            "vixy",
        ]
    ].tail(10)
)

         date     taiex  usd_twd     spy     qqq    soxx    tlt    uup   vixy
45 2025-03-18  49330.94   33.000  556.22  479.20  202.65  84.74  27.35  48.49
46 2025-03-19  48642.44   33.045  550.21  471.03  199.97  84.82  27.33  49.83
47 2025-03-20  49577.90   33.010  556.20  477.34  201.59  85.26  27.39  48.13
48 2025-03-21  49206.44   33.000  554.59  475.72  199.99  85.32  27.47  47.33
49 2025-03-24  48979.66   33.035  554.78  477.29  197.89  84.81  27.56  47.36
50 2025-03-25  49349.28   33.080  564.71  487.76  203.77  83.94  27.61  44.31
51 2025-03-26  49321.23   33.095  566.07  490.54  202.66  83.93  27.58  44.67
52 2025-03-27  48649.66   33.110  559.31  481.52  196.99  83.38  27.70  46.03
53 2025-03-28  47876.79   33.100  557.83  478.77  193.11  83.14  27.62  46.33
54 2025-03-31  45866.82   33.205  546.59  466.17  187.27  84.29  27.57  50.48


In [5]:
REGIME_FEATURE_COLUMNS = [
    # Taiwan market
    "taiex_ret_5d",
    "taiex_ret_20d",
    "taiex_vol_20d",

    # US equity / tech
    "spy_ret_5d",
    "spy_ret_20d",
    "qqq_ret_5d",
    "qqq_ret_20d",
    "soxx_ret_5d",
    "soxx_ret_20d",

    # Rates proxy
    "tlt_ret_5d",
    "tlt_ret_20d",

    # FX
    "usd_twd_ret_5d",
    "usd_twd_ret_20d",

    # Volatility proxy
    "vixy_ret_5d",
    "vixy_ret_20d",
]
features = service.add_price_features(panel)

print(
    features[
        ["date"] + REGIME_FEATURE_COLUMNS
    ].tail()
)

         date  taiex_ret_5d  taiex_ret_20d  taiex_vol_20d  spy_ret_5d  \
50 2025-03-25      0.037177      -5.294453      16.804690    1.526374   
51 2025-03-26      1.395469      -4.211780      16.459130    2.882536   
52 2025-03-27     -1.872286      -5.987762      16.717676    0.559151   
53 2025-03-28     -2.702187      -6.079135      16.811253    0.584215   
54 2025-03-31     -6.355373      -8.848073      21.563915   -1.476261   

    spy_ret_20d  qqq_ret_5d  qqq_ret_20d  soxx_ret_5d  soxx_ret_20d  \
50    -4.022910    1.786311    -6.593386     0.552677     -8.219980   
51    -3.351545    4.141987    -4.939635     1.345202     -6.526452   
52    -4.028895    0.875686    -5.497223    -2.281859     -7.228972   
53    -4.330452    0.641133    -6.263216    -3.440172    -10.613775   
54    -6.201843   -2.329820    -7.581134    -5.366618     -9.400097   

    tlt_ret_5d  tlt_ret_20d  usd_twd_ret_5d  usd_twd_ret_20d  vixy_ret_5d  \
50   -0.944064     0.490842        0.242424         1.115

In [7]:
from investment_agents.agents.regime import MarketRegimeAgent

feature_cols = REGIME_FEATURE_COLUMNS

latest = (
    features
    .dropna(subset=feature_cols)
    .iloc[-1]
)

regime_data = {
    col: float(latest[col])
    for col in feature_cols
}

agent = MarketRegimeAgent()

report = agent.analyze(regime_data)

print(report)

regime='strong_risk_off' risk_score=10 confidence=90 summary="The current environment is strongly risk-off. Taiwan equities show significant negative momentum with high volatility, indicating market instability. US equities, particularly tech and semiconductors, also show strong negative returns. TLT's decline suggests a lack of safe-haven demand, while rising USD/TWD indicates FX pressure. Increasing VIXY signals higher market stress. Overall signals are consistent in showing a hostile environment for equity risk-taking."


In [8]:
from finmind import FinMindClient

from investment_agents.data.repositories.market_regime import (
    MarketRegimeRepository,
)
from investment_agents.features.regime import (
    RegimeFeatureService,
)
from investment_agents.snapshots.regime import (
    MarketRegimeSnapshotService,
)


client = FinMindClient()

repository = MarketRegimeRepository(client)

feature_service = RegimeFeatureService()

snapshot_service = MarketRegimeSnapshotService(
    repository=repository,
    feature_service=feature_service,
)


snapshot = snapshot_service.get_snapshot(
    as_of_date="2025-03-31"
)

print("=== 2025-03-31 ===")

for key, value in snapshot.items():
    print(f"{key}: {value:.4f}")

2026-09-21 00:12:39.813 | INFO     | FinMind.data.finmind_api:get_data:166 - download Dataset.TaiwanStockTotalReturnIndex, data_id: TAIEX
2026-09-21 00:12:40.003 | INFO     | FinMind.data.finmind_api:get_data:166 - download Dataset.TaiwanStockTotalInstitutionalInvestors, data_id: 


=== 2025-03-31 ===
taiex_ret_5d: -6.3554
taiex_ret_20d: -8.8481
taiex_vol_20d: 21.5639
spy_ret_5d: -1.4763
spy_ret_20d: -6.2018
qqq_ret_5d: -2.3298
qqq_ret_20d: -7.5811
soxx_ret_5d: -5.3666
soxx_ret_20d: -9.4001
tlt_ret_5d: -0.6131
tlt_ret_20d: -2.1704
usd_twd_ret_5d: 0.5146
usd_twd_ret_20d: 0.8504
vixy_ret_5d: 6.5878
vixy_ret_20d: 12.7541


In [9]:
snapshot = snapshot_service.get_snapshot(
    as_of_date="2025-03-30"
)

print(snapshot)

2026-09-21 00:13:00.504 | INFO     | FinMind.data.finmind_api:get_data:166 - download Dataset.TaiwanStockTotalReturnIndex, data_id: TAIEX
2026-09-21 00:13:00.562 | INFO     | FinMind.data.finmind_api:get_data:166 - download Dataset.TaiwanStockTotalInstitutionalInvestors, data_id: 


{'taiex_ret_5d': -2.702186949513119, 'taiex_ret_20d': -6.079135399299307, 'taiex_vol_20d': 16.811253234832886, 'spy_ret_5d': 0.5842153663066396, 'spy_ret_20d': -4.330452082047065, 'qqq_ret_5d': 0.6411334398385593, 'qqq_ret_20d': -6.263215600281935, 'soxx_ret_5d': -3.4401720086004284, 'soxx_ret_20d': -10.61377522680984, 'tlt_ret_5d': -2.5550867323019144, 'tlt_ret_20d': -3.009799346710218, 'usd_twd_ret_5d': 0.3030303030302939, 'usd_twd_ret_20d': 0.853138330286396, 'vixy_ret_5d': -2.112824846820194, 'vixy_ret_20d': 7.245370370370363}
